## p72

In [ ]:
#1: chat model
# Not Free anymore
# from langchain_openai import ChatOpenAI
# llm_model = ChatOpenAI(model_name = "gpt-4o")
from langchain_ollama import ChatOllama
llm_model = ChatOllama(model="llama3.1:latest")



In [52]:
#2: messages
from langchain_core.messages import HumanMessage, AIMessage
messages = [HumanMessage("what is 4+7?")]
# messages = [{"role":"HumanMessage", "content":"What is the weather today?"}]
# messages = [("human", "What is the weather today?")]

In [53]:
#3: define a tool
from langchain_core.tools import tool

@tool
def add_numbers(x: int, y:int) -> int:
    """Adds two integer numbers"""
    return x+y

In [54]:
#4: tool invocation 
llm_with_tools = llm_model.bind_tools([add_numbers])
ai_message = llm_with_tools.invoke(messages)
for tool_call in ai_message.tool_calls:
    tool_response = add_numbers.invoke(tool_call)

In [56]:
ai_message = llm_with_tools.invoke(messages)

print("CONTENT:", ai_message.content)
print("TOOL CALLS:", ai_message.tool_calls)

CONTENT: 
TOOL CALLS: [{'name': 'add_numbers', 'args': {'x': '4', 'y': '7'}, 'id': '31a9307d-962b-4a52-bffb-444e0af1e511', 'type': 'tool_call'}]


## P74

In [79]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_ollama import ChatOllama

@tool
def multiply(a:int, b:int) -> int:
    """Multiply 'a' times 'b'. """
    return a*b
@tool
def exponetiate(a: float, b:float) -> float:
    """Raise 'a' to the 'b'."""
    return a**b
@tool
def add(a:float, b:float)->float:
    """Add 'a' and 'b'."""
    return a+b
tools = [multiply, exponetiate, add]

llm_model = ChatOllama(model="llama3.1:latest").bind_tools(tools)
messages = [HumanMessage("what is 2**4 and also 87+10?")]

output = llm_model.invoke(messages)

messages.append(output)
for tool_call in output.tool_calls:
    selected_tools = {"add":add, "multiply":multiply, "exponetiate":exponetiate}[tool_call["name"].lower()]
    tool_msg = selected_tools.invoke(tool_call)
    print(f"{tool_msg.name} {tool_call['args']} {tool_msg.content}")
    messages.append(tool_msg)
    final_response = llm_model.invoke(messages)
    print(final_response.content)

exponetiate {'b': 4, 'a': 2} 16.0
The result of 2**4 is 16.

And, the result of 87+10 is 97.
add {'a': 87, 'b': 10} 97.0
The result of `2**4` is 16, and the result of `87+10` is 97.


## P76


In [21]:
# Wikipedia API call
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

#API call and wrap as tool
api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=300)
tool = WikipediaQueryRun(api_wrapper=api_wrapper)

#The LLM model
llm_model = ChatOllama(model="llama3.1:latest").bind_tools([tool])
messages = [HumanMessage("what is th most impressive thing about Buzz Aldrin (Dr. Rendezvous)")]

output = llm_model.invoke(messages)
messages.append(output)

for tool_call in output.tool_calls:
    tool_msg = tool.invoke(tool_call)
    print(tool_msg.name)
    print(tool_call['args'])
    print(tool_msg.content)
    messages.append(tool_msg)
    print()
final_response = llm_model.invoke(messages)
print(final_response.content)

wikipedia
{'query': 'Buzz Aldrin'}
Page: Buzz Aldrin
Summary: Buzz Aldrin ( AWL-drin; born Edwin Eugene Aldrin Jr.; January 20, 1930) is an American former astronaut, aeronautical engineer, and fighter pilot. He was the second person to walk on the Moon after mission commander Neil Armstrong. He made three spacewalks as pilot of the 

The most impressive thing about Buzz Aldrin (Dr. Rendezvous) is that he was the second person to walk on the Moon during the Apollo 11 mission, a historic achievement in space exploration and one of the greatest accomplishments of human kind.


In [12]:
# Stock market API

from langchain_core.tools import tool
import yfinance as yf
import requests
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

@tool
def get_stock_price(ticker: str) -> str:
    """get the stock price for the stock exchange ticker for the company."""
    data = yf.Ticker(ticker)
    price = data.fast_info.last_price
    return str(price)
# def get_stock_price(ticker: str) -> str:
#     """get the stock price for the stock exchange ticker for the company."""
#     api_url = f"https://api.example.com/stocks/{ticker}"
#     response=requests.get(api_url)
#     if response.status_code==200:
#         data = response.json()
#         return data["price"]
#     else:
#         raise ValueError(f"failed to fetch stock price for {ticker}")

#model
llm_mdoel = ChatOllama(model="llama3.1:latest").bind_tools([get_stock_price])
messages = [HumanMessage("What is the stock price of Apple?")]
output = llm_mdoel.invoke(messages)
messages.append(output)

for tool_call in output.tool_calls:
    tool_msg = get_stock_price.invoke(tool_call)

    print(tool_msg.name)
    print(tool_call["args"])
    print(tool_msg.content)
    messages.append(tool_msg)

final_response =llm_mdoel.invoke(messages)
print(final_response.content)



get_stock_price
{'ticker': 'AAPL'}
255.9199981689453
The current stock price of Apple (AAPL) is approximately $255.92.
